# The Analysis of Token Level Probabilities
Now that I have computed the token-level probabilities, I want to assess their range and the differences (if any) between my chosen programming languages. 

In [1]:
import os
import pandas as pd

In [2]:
python_csv_file = '/datasets/probabilities_python_3.csv'
c_sharp_csv_file = '/datasets/probabilities_c_sharp_3.csv'
typescript_csv_file = '/datasets/probabilities_typescript_3.csv'

def get_prob_file(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_probs = get_prob_file(python_csv_file)
c_sharp_probs = get_prob_file(c_sharp_csv_file)
typescript_probs = get_prob_file(typescript_csv_file)

In [3]:
python_probs = python_probs.loc[python_probs['line_number'] != 0]

In [4]:
python_probs.head(29)

,title,line_number,line_text,token_prob
1,Sports Rank,1,def competition_ranking(scores):\n,0.927395
2,Sports Rank,2,n = len(scores)\n,0.925640
3,Sports Rank,3,rank = [0] * n\n,0.864678
4,Sports Rank,4,"sorted_indices = sorted(range(n), key=lamb...",0.764331
5,Sports Rank,5,current_rank = 1\n,0.821992
6,Sports Rank,6,for i in range(n):\n,0.879384
7,Sports Rank,7,if i > 0 and scores[sorted_indices[i]]...,0.890043
8,Sports Rank,8,current_rank = i + 1\n,0.997475
9,Sports Rank,9,rank[sorted_indices[i]] = current_rank\n,0.999960
10,Sports Rank,10,return rank\n\n,0.999986


## Quality Check
Have all lines a computed probability?

In [13]:
python_csv_file = '/datasets/selected_problems_python_14B.csv'
c_sharp_csv_file = '/datasets/selected_problems_c_sharp_14B.csv'
typescript_csv_file = '/datasets/selected_problems_typescript_14B.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [14]:
python_probs['line_text'] = python_probs['line_text'].str.strip()
c_sharp_probs['line_text'] = c_sharp_probs['line_text'].str.strip()
typescript_probs['line_text'] = typescript_probs['line_text'].str.strip()

def check_line_probs(df, probs, comment):
    for index, row in df.iterrows():
        lines = row['solution'].split('\n')
        for line in lines:
            line = line.strip()
            matched_rows = probs.loc[(probs['line_text'] == line) & (probs['title'] == row['title'])]
            if matched_rows.empty and not line.startswith(comment) and not line.isspace() and line != '':
                print(row)
                print(f"Line: {line}")

In [15]:
check_line_probs(python_solutions, python_probs, '#')

In [16]:
check_line_probs(c_sharp_solutions, c_sharp_probs, '//')

In [17]:
check_line_probs(typescript_solutions, typescript_probs, '//')

# Overall Analysis

In [8]:
def get_stats(df):
    overall_min = df['token_prob'].min()
    overall_max = df['token_prob'].max()
    overall_stats = df['token_prob'].agg(['mean', 'median', 'std'])
    return overall_min, overall_max, overall_stats

In [9]:
stats = get_stats(python_probs)

print(f"Min: {stats[0]}, Max: {stats[1]}")
print("Overall token_prob stats:")
print(stats[2])

Min: 0.2751128673553467, Max: 0.9999998807907104
Overall token_prob stats:
mean      0.944865
median    0.990878
std       0.094931
Name: token_prob, dtype: float64


In [10]:
stats = get_stats(c_sharp_probs)

print(f"Min: {stats[0]}, Max: {stats[1]}")
print("Overall token_prob stats:")
print(stats[2])

Min: 0.2108441591262817, Max: 1.0
Overall token_prob stats:
mean      0.969782
median    0.999767
std       0.076648
Name: token_prob, dtype: float64


In [11]:
stats = get_stats(typescript_probs)

print(f"Min: {stats[0]}, Max: {stats[1]}")
print("Overall token_prob stats:")
print(stats[2])

Min: 0.3603237271308899, Max: 1.0
Overall token_prob stats:
mean      0.960418
median    0.998737
std       0.080108
Name: token_prob, dtype: float64


In [20]:
x = 0.5
total_python_rows = len(python_probs)
count = (python_probs['token_prob'] < x).sum()
print(f"Number of values lower than {x}: {count} out of {total_python_rows}")

Number of values lower than 0.5: 2 out of 418


# Difficulty-Level Analysis
I also need the selected problems to make a comparison between the probabilties based on the difficulty level.

In [22]:
csv_file = '/datasets/selected_problems.csv'
csv_full_path = os.getcwd()+csv_file
if os.path.exists(csv_full_path):
    selected_problems = pd.read_csv(csv_full_path)
else:
    selected_problems = None
    print("File not found.")

In [23]:
selected_problems.head()

,title,description,difficulty,source
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis
1,Solidus Fraction,A fraction written with a solidus (/) can be w...,Medium,Kattis
2,Breakout,A variant of the popular video game breakout u...,Medium,Kattis
3,Passport Dates,Konráð has managed to obtain a big pile of Ice...,Easy,Kattis
4,Weather Forecast,One of the most important things you can do in...,Hard,Kattis


In [24]:
def merge_dfs(df):
    python_probs_merged = pd.merge(df, selected_problems, on='title')
    python_probs_merged = python_probs_merged.drop(['line_number', 'line_text', 'description', 'title'], axis=1)
    return python_probs_merged

In [25]:
python_probs_merged = merge_dfs(python_probs)
python_probs_merged.head()

,token_prob,difficulty,source
0,0.927395,Medium,Kattis
1,0.925640,Medium,Kattis
2,0.864678,Medium,Kattis
3,0.764331,Medium,Kattis
4,0.821992,Medium,Kattis


In [29]:
range_by_difficulty_python = python_probs_merged.groupby('difficulty')['token_prob'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_python)

                 min  max      mean    median       std
difficulty                                             
Easy        0.275113  1.0  0.943536  0.993856  0.109548
Hard        0.548505  1.0  0.939697  0.981253  0.090369
Medium      0.483016  1.0  0.949651  0.994434  0.091744


In [30]:
c_sharp_probs_merged = merge_dfs(c_sharp_probs)
typescript_probs_merged = merge_dfs(typescript_probs)

In [31]:
range_by_difficulty_c_sharp = c_sharp_probs_merged.groupby('difficulty')['token_prob'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_c_sharp)

                 min  max      mean    median       std
difficulty                                             
Easy        0.580035  1.0  0.970127  0.999864  0.064298
Hard        0.660657  1.0  0.970270  0.999431  0.061516
Medium      0.210844  1.0  0.969161  0.999858  0.093235


In [32]:
range_by_difficulty_typescript = typescript_probs_merged.groupby('difficulty')['token_prob'].agg(['min', 'max', 'mean', 'median', 'std'])
print(range_by_difficulty_typescript)

                 min  max      mean    median       std
difficulty                                             
Easy        0.557620  1.0  0.960415  0.990732  0.069996
Hard        0.613514  1.0  0.968954  0.999854  0.063354
Medium      0.360324  1.0  0.947418  0.995042  0.103674
